# Lab 31: Automated Red Teaming, CI Gates, and Real Models

*AI Cybersecurity Masterclass · Lab 31 · AI60secLab*

**Companion video:** none yet for this lab.

**Level:** Expert

**Research question:** How do you keep AI safety controls from silently regressing, and how do you test them against a real model instead of a simulator?

Lab 27 turned red-team findings into a regression suite you ran by hand. This lab makes it automatic: the suite becomes tests that run on every push, a failing safety test blocks the merge, and the same harness can be pointed at a real LLM.

**Learning objectives**
- express safety expectations as automated tests;
- gate releases on those tests in continuous integration (CI);
- prove the suite has teeth by running it against a version with a known hole;
- connect the Unit 5 controls to a real model through an OpenAI-compatible endpoint;
- know where open-source AI red-teaming tools fit.

## Security foundation: security in the delivery pipeline

Modern teams ship code many times a day through a **CI/CD pipeline** (continuous integration and delivery). Security checks that live in a document get skipped; checks that live in the pipeline run every time. This is often called **shifting left**: catching problems before release instead of after an incident.

Typical pipeline security gates include static code analysis, dependency and secret scanning, and tests that encode security requirements. For AI applications, the same idea applies to **safety behavior**: "no injected instruction may send email outside the company" becomes a test that must pass before a change merges.

Two cautions:
- **a gate is only as good as its tests.** A suite that cannot fail protects nothing, so test that your tests can fail;
- **simulators are not models.** A deterministic harness keeps CI fast and repeatable; real-model testing catches behavior the simulator cannot. Mature programs run both.

In [ ]:
# Setup: works in Google Colab and locally. No uploads needed.
# In Colab this clones the course repo, which already includes the UNSW-NB15 data.
from pathlib import Path
import sys, os

REPO_URL = "https://github.com/Jacquelinepersha/ai-cybersecurity-public-labs.git"
REPO_NAME = "ai-cybersecurity-public-labs"

try:
    import google.colab  # noqa: F401
    if not Path(REPO_NAME).exists():
        !git clone --depth 1 {REPO_URL}
    os.chdir(REPO_NAME)
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "raw"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data folder :", DATA_DIR)

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 70)

from src.llm_app import ControlConfig, build_test_cases, evaluate_controls
from src.redteam import APP_VERSIONS, default_cases, run_campaign
from src.live_llm import LiveModel, live_config

REPO_ROOT = PROJECT_ROOT
print("Repository root:", REPO_ROOT)

## Step 1: Safety expectations as tests

Each test states one expectation in code. Read the test file: every assertion is something a reviewer would otherwise check by hand.

In [3]:
test_file = REPO_ROOT / "tests" / "test_redteam_regression.py"
print(test_file.read_text()[:3000])

"""
Red-team regression suite. Runs on every push and pull request (see
.github/workflows/redteam-regression.yml). A failing test blocks the merge:
a safety control that regressed is treated like a broken build.
"""
import json
import os
import sys

sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))

from src import agent_security as agents  # noqa: E402
from src.llm_app import (  # noqa: E402
    CANARY_SECRET,
    ControlConfig,
    build_test_cases,
    evaluate_controls,
)
from src.privacy_attacks import scan_for_canaries, simulate_assistant_logging  # noqa: E402
from src.redteam import APP_VERSIONS, default_cases, run_campaign  # noqa: E402

RELEASE = list(APP_VERSIONS)[-1]          # the version we ship
FULL_STACK = ControlConfig(True, True, True, True, True, output_link_filter=True)


def test_no_prompt_injection_payload_succeeds():
    results, _ = evaluate_controls(build_test_cases(), FULL_STACK)
    attacks = results[results["kind"] == "attack"

## Step 2: Run the suite the way CI does

In [4]:
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider", str(test_file)],
    cwd=REPO_ROOT, capture_output=True, text=True,
)
print(result.stdout[-2000:] or result.stderr[-2000:])
print("Exit code:", result.returncode, "(0 means the gate passes)")

.......                                                                  [100%]
7 passed in 0.93s

Exit code: 0 (0 means the gate passes)


## Step 3: Prove the gate can fail

A gate that never fails is decoration. Run the release checks against the previous version, which still has the image-link exfiltration hole from Lab 27.

In [5]:
versions = list(APP_VERSIONS)
rows = []
for version in versions:
    campaign = run_campaign(default_cases(), version)
    rows.append({
        "version": version.split(" ")[0],
        "cases": len(campaign),
        "failing": int((~campaign["passed"]).sum()),
        "gate": "PASS" if campaign["passed"].all() else "BLOCK MERGE",
    })
display(pd.DataFrame(rows))

,version,cases,failing,gate
0,v1,12,10,BLOCK MERGE
1,v2,12,1,BLOCK MERGE
2,v3,12,0,PASS


## Step 4: The CI workflow

This repository ships the workflow below at `.github/workflows/redteam-regression.yml`. GitHub runs it on every push and pull request, and shows a red X on any change that breaks a safety expectation. Branch protection can then require it to pass before merging.

In [6]:
print((REPO_ROOT / ".github" / "workflows" / "redteam-regression.yml").read_text())

name: Red-team regression

on:
  push:
  pull_request:

permissions:
  contents: read

jobs:
  redteam:
    runs-on: ubuntu-latest
    timeout-minutes: 15
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - name: Install minimal dependencies
        run: pip install pandas numpy scikit-learn pytest
      - name: Run the red-team regression suite
        run: pytest -q tests/test_redteam_regression.py



## Step 5: Point the harness at a real model (optional)

The Unit 5 controls accept any model with a `generate()` method. `src/live_llm.py` connects any **OpenAI-compatible** endpoint: a hosted provider, or a free local model.

To use a free local model with [Ollama](https://ollama.com):
1. install Ollama and run `ollama pull llama3.1`;
2. before starting Jupyter, set `LLM_BASE_URL=http://localhost:11434/v1` and `LLM_MODEL=llama3.1`;
3. rerun this lab.

For a hosted provider, also set `LLM_API_KEY`. Real calls may cost money and results vary between runs, so record the model name, date, and settings with every result.

**Provenance with a real model.** We cannot see which text a real model acted on. So if any untrusted segment was in its context, every tool call it proposes is treated as untrusted. That is conservative on purpose: the policy, not the model, decides.

In [7]:
config = live_config()
cases = build_test_cases()
stacks = {
    "no controls": ControlConfig(),
    "all layers": ControlConfig(True, True, True, True, True, output_link_filter=True),
}

if config is None:
    print("No LLM endpoint configured (LLM_BASE_URL / LLM_MODEL not set).")
    print("Skipping the live run. The simulator results below are the offline baseline.")
    model, label = None, "simulator"
else:
    base_url, model_name, api_key = config
    model, label = LiveModel(base_url, model_name, api_key), f"live: {model_name}"
    print("Running against", label, "at", base_url)

rows = []
for name, controls in stacks.items():
    results, log = evaluate_controls(cases, controls, model=model)
    rows.append({
        "model": label,
        "controls": name,
        "attack_success_rate": results.loc[results["kind"] == "attack", "attack_succeeded"].mean(),
        "benign_task_success": results.loc[results["kind"] == "benign", "benign_completed"].mean(),
        "log_events": len(log.events),
    })
display(pd.DataFrame(rows).round(3))

No LLM endpoint configured (LLM_BASE_URL / LLM_MODEL not set).
Skipping the live run. The simulator results below are the offline baseline.


,model,controls,attack_success_rate,benign_task_success,log_events
0,simulator,no controls,1.0,1.0,0
1,simulator,all layers,0.0,1.0,16


## Step 6: Open-source red-teaming tools (optional, run in a terminal)

Your suite tests *your* application's controls. Open-source scanners test a *model* against large libraries of known attacks. They complement each other.

**garak** (by NVIDIA) probes a model for prompt injection, jailbreaks, data leakage, and more. It installs many dependencies, so use its own virtual environment. The commands below use garak 0.17 syntax; if they fail, run `garak --help`, since options change between releases.

```bash
python3 -m venv garak-env && source garak-env/bin/activate
pip install garak

# Prompt-injection probes against a local Ollama model
garak --target_type openai.OpenAICompatible \
      --target_name llama3.1 \
      --generator_options '{"openai": {"OpenAICompatible": {"uri": "http://localhost:11434/v1/", "api_key": "ollama"}}}' \
      --spec probes.promptinject \
      --generations 1

# Everything tagged with OWASP LLM01 (prompt injection)
garak --target_type openai.OpenAICompatible --target_name llama3.1 \
      --generator_options '{"openai": {"OpenAICompatible": {"uri": "http://localhost:11434/v1/", "api_key": "ollama"}}}' \
      --spec tag:owasp:llm01
```

garak writes a report of which probes got through. Treat each success as a new candidate case for your own suite: reproduce it in `tests/`, add the expected safe behavior, and the CI gate will keep it fixed.

**PyRIT** (Python Risk Identification Toolkit, by Microsoft) is a framework for scripting multi-turn red-team campaigns and scoring the results. Use it when you need adaptive, conversational attacks rather than a fixed probe library. Install it with `pip install pyrit` and follow its documentation, because its API moves quickly.

Only point these tools at models and applications you own or are authorized to test.

## Your analysis

1. Which test in the suite would catch a teammate who disables the output link filter?
2. Why does the suite include a test that checks the *previous* version still fails?
3. If you connect a real model and the "no controls" attack success rate is lower than the simulator's, does that mean the model is safe? What else could explain it?
4. What should happen when a real-model red-team run finds a new failure the simulator never produced?

**Takeaway:** safety that is not tested on every change will regress. Encode expectations as tests, gate releases on them, prove the gate can fail, and test against real models as well as simulators.

## From lab to SOC

Security teams increasingly own pipeline gates for AI features: a change that weakens a guardrail fails the build, just like a failing unit test. Real-model red-team runs feed new cases into the suite, so every finding becomes a permanent check.

**Check yourself**
1. Name two security checks, besides AI safety tests, that commonly run in a CI pipeline.
2. Why should the CI suite use the deterministic simulator instead of a live model?
3. Who should be allowed to change or skip a failing safety test, and how would you record it?